# コイン集めアリーナ — クラス対戦演習

これまでの発展教材（クラス → 簡単なゲーム → バイナリとセキュリティ → ネットワークプログラミング）の 総まとめ です。

教室のみなさんのPython botが、1つの盤面で同時に競います。

## ゲームのルール

- 盤面（24×24マス）に コイン がいくつか出現します
- 各プレイヤーは、上下左右に1マスずつ動けます
- コインの上に乗ると +1点。コインは別の場所に再出現します
- 制限時間で、得点が一番高い人の勝ち

## 2つの役割

- 先生: サーバ（審判＋盤面）を立て、公開URL を配り、盤面をスクリーンに映す
- 生徒: サーバに接続する bot を書く。良い戦略を考えた人が勝つ

使う技術は全部これまでの復習です：HTTPクライアント/サーバ・JSON・bot（ネットワーク回）、最後の発展で ハッシュ／署名（バイナリ回）も出てきます。

---
# 共通: アリーナのコード

次の 2つのセル（盤面の見た目 と サーバ本体）は、先生、および ソロ練習する生徒 が実行します。アリーナを動かすための土台です。

まず下のセルで、盤面の見た目（JavaScript）を用意します。折りたたまれています。中身のJS（盤面の見た目）に興味があれば、タイトルをクリックすると展開できます。

In [ ]:
# 盤面の見た目
# 盤面のHTML/JSを board.html に書き出す（サーバがこれを配信する）
board_html = r'''
<!doctype html><html><head><meta charset="utf-8"><title>コイン集めアリーナ</title>
<style>
 body{background:#0d1117;color:#e6edf3;font-family:sans-serif;text-align:center;margin:0}
 canvas{background:#161b22;border-radius:8px;margin-top:10px}
 #rank{display:inline-block;text-align:left;margin:8px auto;font-size:15px}
</style></head>
<body>
 <h2>コイン集めアリーナ</h2>
 <canvas id="c" width="480" height="480"></canvas>
 <div id="rank"></div>
 <script>
   const cell = 20;
   const ctx = document.getElementById('c').getContext('2d');
   async function tick() {
     let s;
     try { s = await (await fetch('/state')).json(); } catch (e) { return; }
     ctx.clearRect(0, 0, 480, 480);
     // 格子線
     ctx.strokeStyle = '#2a2f3a';
     ctx.lineWidth = 1;
     for (let i = 0; i <= 24; i++) {
       ctx.beginPath(); ctx.moveTo(i * cell, 0); ctx.lineTo(i * cell, 480); ctx.stroke();
       ctx.beginPath(); ctx.moveTo(0, i * cell); ctx.lineTo(480, i * cell); ctx.stroke();
     }
     // コイン（黄色）
     ctx.fillStyle = '#f1c40f';
     for (const c of s.coins) {
       ctx.beginPath();
       ctx.arc(c[0] * cell + cell / 2, c[1] * cell + cell / 2, 6, 0, 7);
       ctx.fill();
     }
     // プレイヤー（各自の色）
     for (const p of s.players) {
       ctx.fillStyle = p.color;
       ctx.beginPath();
       ctx.arc(p.x * cell + cell / 2, p.y * cell + cell / 2, 8, 0, 7);
       ctx.fill();
       ctx.fillStyle = '#fff';
       ctx.font = '10px sans-serif';
       ctx.fillText(p.name, p.x * cell - 4, p.y * cell - 4);
     }
     // ランキング
     const r = [...s.players].sort((a, b) => b.score - a.score);
     document.getElementById('rank').innerHTML = '<b>ランキング</b><br>' +
       r.map((p, i) => `${i + 1}. <span style="color:${p.color}">■</span> ${p.name}: ${p.score}`).join('<br>');
   }
   setInterval(tick, 150);
 </script>
</body></html>
'''
with open("board.html", "w", encoding="utf-8") as f:
    f.write(board_html)
print("board.html を用意しました。")

次がサーバ本体です。ネットワーク回で学んだ `http.server` の応用なので、興味があれば読んでみてください。

In [ ]:
# === アリーナサーバ（ネットワーク回で学んだ http.server の応用）===
# ※ 先に「board.html を書き出すセル」を実行しておくこと
import json, threading, time, random
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

W, H, NUM_COINS = 24, 24, 6

def _rand_pos(): return [random.randint(0, W-1), random.randint(0, H-1)]

class ArenaServer(ThreadingHTTPServer):
    """世界の状態（プレイヤー・コイン）を、このサーバ自身が持つ。
       別サーバ（本番とソロ練習）は状態を共有しない。"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lock = threading.Lock()
        self.players = {}
        self.coins = [_rand_pos() for _ in range(NUM_COINS)]
        self.next_id = 0
    def reset(self):
        with self.lock:
            self.players = {}
            self.coins = [_rand_pos() for _ in range(NUM_COINS)]
            self.next_id = 0

class Arena(BaseHTTPRequestHandler):
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Access-Control-Allow-Origin", "*")   # どこからでも接続可
        self.end_headers(); self.wfile.write(b)
    def _body(self):
        n = int(self.headers.get("Content-Length", 0))
        return json.loads(self.rfile.read(n) or b"{}")

    def do_GET(self):
        srv = self.server
        if self.path == "/" or self.path.startswith("/?"):
            with open("board.html", encoding="utf-8") as f:
                body = f.read().encode()
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers(); self.wfile.write(body)
        elif self.path == "/state":
            with srv.lock:
                self._send({"w": W, "h": H, "coins": srv.coins,
                    "players": [{"id": i, "name": p["name"], "x": p["x"], "y": p["y"],
                                 "score": p["score"], "color": p["color"]} for i, p in srv.players.items()]})
        else:
            self._send({"error": "not found"}, 404)

    def do_POST(self):
        srv = self.server
        d = self._body()
        if self.path == "/join":
            with srv.lock:
                i = srv.next_id; srv.next_id += 1; pos = _rand_pos()
                srv.players[i] = {"name": str(d.get("name", "noname"))[:12], "x": pos[0], "y": pos[1],
                                  "score": 0, "color": "#%06x" % random.randint(0x333333, 0xffffff)}
                self._send({"id": i, "w": W, "h": H})
        elif self.path == "/move":
            with srv.lock:
                p = srv.players.get(d.get("id"))
                if not p: self._send({"error": "unknown id"}, 400); return
                dx, dy = {"up": (0,-1), "down": (0,1), "left": (-1,0), "right": (1,0)}.get(d.get("dir"), (0,0))
                p["x"] = max(0, min(W-1, p["x"] + dx)); p["y"] = max(0, min(H-1, p["y"] + dy))
                for c in srv.coins:
                    if c[0] == p["x"] and c[1] == p["y"]:
                        p["score"] += 1; c[0], c[1] = _rand_pos()
                self._send({"x": p["x"], "y": p["y"], "score": p["score"]})
        elif self.path == "/reset":
            srv.reset()
            self._send({"ok": True})
        else:
            self._send({"error": "not found"}, 404)

    def log_message(self, *a): pass

def start_server():
    srv = ArenaServer(("0.0.0.0", 0), Arena)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    time.sleep(0.3)
    return srv, srv.server_address[1]

print("アリーナの準備OK。start_server() で起動できます。")

---
# A. 先生セクション

先生だけが実行します。サーバを起動し、公開URLを発行し、盤面をスクリーンに映します。

In [ ]:
# 【先生】サーバを起動する
server, PORT = start_server()
print("サーバ起動 ポート:", PORT)

## 公開URLを発行する

`cloudflared` で、教室の全員が接続できる 公開URL（`https://xxxx.trycloudflare.com`）を発行します。

In [ ]:
# 公開URLを発行（cloudflared）
# ※ ネットワーク環境によっては使えないことがあります。その場合は「ソロ練習」で開発を。
import os, re, subprocess, urllib.request

if not os.path.exists("cloudflared"):
    print("cloudflared をダウンロード中...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:%d" % PORT],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in proc.stdout:                      # 出力から公開URLを拾う
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break
print("=" * 50)
print("公開URL:", public_url)
print("=" * 50)

## ラウンドをリセットする

対戦を仕切り直すとき（全員の得点・位置を消して新しいコインにする）に実行します。

In [ ]:
# 【先生】ラウンドをリセット
import requests
requests.post("http://127.0.0.1:{}/reset".format(PORT))
print("アリーナをリセットしました（新しいラウンド）")

---
# B. 生徒セクション

各自のColabで、サーバに接続する bot を作ります。

## APIの仕様

| やりたいこと | 呼び出し | 返り値 |
|---|---|---|
| 参加する | `POST /join` `{"name": "名前"}` | `{"id": 自分の番号, "w":24, "h":24}` |
| 今の盤面を見る | `GET /state` | `{"coins": [[x,y]...], "players": [{"id","name","x","y","score","color"}...]}` |
| 動く | `POST /move` `{"id": 自分の番号, "dir": 向き}` | `{"x","y","score"}` |

- 向き `dir` は `"up"` / `"down"` / `"left"` / `"right"`
- 座標は x=右方向、y=下方向（左上が 0,0）

## 接続設定

サーバとのやり取りは、ネットワーク回で習った `requests` で行います。使う API は3つだけです。

- 参加する: `requests.post(SERVER + "/join", json={"name": 名前}).json()`
- 盤面を見る: `requests.get(SERVER + "/state").json()`
- 動く: `requests.post(SERVER + "/move", json={"id": 自分のid, "dir": 向き}).json()`

まず接続先の変数 `SERVER` を用意します。

In [ ]:
import requests

# 接続先。この後の「ローカルで練習」のセルで自動的に設定されます。
SERVER = ""

## まずローカルで練習する

本番（先生の公開URL）につなぐ前に、自分のColabの中にアリーナを立てて練習します。下のセルを実行すると、接続先 `SERVER` が自分のローカルサーバに変わります。

In [ ]:
# ソロ練習の準備：サーバを立てる
# 先に「board.html」「アリーナサーバ」「接続設定」のセルを実行しておくこと
import time, requests

srv_local, solo_port = start_server()
SERVER = "http://127.0.0.1:%d" % solo_port   # 接続先を自分のローカルサーバに切り替え
print("ローカルのアリーナ:", SERVER)

# 盤面をノート内に表示（Colab専用）
try:
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(solo_port, path="/")
except Exception:
    pass

## join：ゲームに参加する

サーバに参加すると、自分の `id` がもらえます。これから、この id で「自分」を指定します。まず例題を実行し、次の演習を自分で書きましょう。

In [ ]:
# 例題：参加して、自分の id をもらう
me = requests.post(SERVER + "/join", json={"name": "テスト"}).json()
print(me)                 # 例: {'id': 0, 'w': 24, 'h': 24}
my_id = me["id"]
print("あなたの id:", my_id)

In [ ]:
# 演習：好きな名前で参加して、自分の id を表示しよう



## state：今の盤面を見る

`GET /state` で、コインの位置と全プレイヤーの状態が返ります。

- `state["coins"]` … `[[x, y], ...]`
- `state["players"]` … `[{"id","name","x","y","score","color"}, ...]`

In [ ]:
# 例題：今の盤面を取得して中身を見る
state = requests.get(SERVER + "/state").json()
print("コインの数:", len(state["coins"]))
print("プレイヤー:", state["players"])

In [ ]:
# 演習：state を取得して、今この盤面に何人プレイヤーがいるか表示しよう



## move：動く

自分の `id` と向き（`"up"` / `"down"` / `"left"` / `"right"`）を送ると、その方向へ1歩動きます。コインに乗ると得点です。

In [ ]:
# 例題：右に1歩動く
result = requests.post(SERVER + "/move", json={"id": my_id, "dir": "right"}).json()
print(result)             # 例: {'x': 11, 'y': 5, 'score': 0}

In [ ]:
# 演習：for ループで、好きな向きに5歩動かしてみよう



## botを作る

join / state / move を組み合わせると、自動で動く bot になります。いきなり全部は大変なので、部品を2つ練習してから、最後に while のくり返しにまとめます。

### 部品1：一番近いコインを探す

盤面のコインの中から、自分に一番近いものを選びます。for で1つずつ距離を見て、最小のものを覚えます。距離は `abs(x差) + abs(y差)`。

In [ ]:
# 参加して、今の盤面を取得（自分を探すところまで用意しています）
me = requests.post(SERVER + "/join", json={"name": "あなた"}).json()
my_id = me["id"]
state = requests.get(SERVER + "/state").json()

me_now = None
for p in state["players"]:
    if p["id"] == my_id:
        me_now = p

# 演習：state["coins"] の中から、me_now に一番近いコインを探して best に入れて表示しよう


print("一番近いコイン:", best)

### 部品2：向きを決める

目標のコインの位置と自分の位置を比べて、動く向き（`"up"`/`"down"`/`"left"`/`"right"`）を決めます。

In [ ]:
# 例として、自分の位置とコインの位置を置いておきます
mx, my = 10, 10
cx, cy = 15, 8

# 演習：(mx, my) から (cx, cy) へ向かう向きを direction に入れて表示しよう


print("向き:", direction)

### まとめ：botにする

部品1と部品2を、くり返し（while）の中に組み込みます。下の骨組みの空いている2か所に、上で書いたコードを入れて完成させましょう（15秒プレイ）。

In [ ]:
import time

me = requests.post(SERVER + "/join", json={"name": "あなた"}).json()
my_id = me["id"]

end = time.time() + 15
while time.time() < end:
    state = requests.get(SERVER + "/state").json()

    me_now = None
    for p in state["players"]:
        if p["id"] == my_id:
            me_now = p

    # ここに「部品1：一番近いコインを探す」を書く → best に入れる


    # ここに「部品2：向きを決める」を書く → direction に入れる


    requests.post(SERVER + "/move", json={"id": my_id, "dir": direction})
    time.sleep(0.1)

# 得点を表示
score = 0
for p in requests.get(SERVER + "/state").json()["players"]:
    if p["id"] == my_id:
        score = p["score"]
print("15秒プレイ終了。得点:", score)

## botを強くする

上で作ったbotをコピーして、自分なりに強くしてみましょう。たとえば:

- 他プレイヤーがすぐ近くにいるコインは避ける（`state["players"]` を見る）
- コインが密集している方を狙う

くり返しの中の「向きを決める部分」を工夫します。

In [ ]:
# 上の例題をコピーして、向きの決め方を工夫しよう



## 対戦相手（ダミー敵）を出す

自分のbotができたら、対戦相手を用意しましょう。下のセルは、ランダムに動くだけの敵を3体、`threading` で同時に動かします。実行してから自分のbotをもう一度動かすと、対戦になります。

In [ ]:
import threading, time, random, requests

# ランダムに動くだけの敵
def dummy(name):
    m = requests.post(SERVER + "/join", json={"name": name}).json()
    mid = m["id"]
    while True:
        requests.post(SERVER + "/move", json={"id": mid, "dir": random.choice(["up", "down", "left", "right"])})
        time.sleep(0.15)

for nm in ["敵A", "敵B", "敵C"]:
    threading.Thread(target=dummy, args=(nm,), daemon=True).start()
print("ダミーの敵を3体 動かしました。")

---
## 本番：クラス対戦

完成した自分のbotで、先生の公開URLにつないで全員と競います。下のセルで接続先を本番に変えてから、自分のbotのセルをもう一度実行しましょう。

In [ ]:
# 本番：先生の公開URLに変えて、自分のbotで参加する
SERVER = "https://ここに先生のURL.trycloudflare.com"

# SERVER を変えたら、上で作った自分のbotのセルをもう一度実行すると本番に参加できます。
# 対戦時間に合わせて、bot の中の end = time.time() + 15 の 15 を長くしてもOKです。
print("接続先を本番に変えました。自分のbotのセルを実行しましょう。")

---
# 不正（チート）とその対策

このアリーナは、実は id さえ知っていれば他人のプレイヤーも動かせて しまいます。通信は前回学んだとおり HTTPで丸見え なので、覗いて真似すれば「なりすまし移動」ができてしまうのです。

「不正ができてしまう」と分かるのは、仕組みを理解できた証拠。では、どう防ぐか——前回の ハッシュ／デジタル署名 の出番です。

---
# 発展: 人間プレイヤーとして参加する（Gradio操作パネル）

これまでプレイヤーは bot（プログラム） でした。前回学んだ Gradio で 操作パネル を作れば、人間がボタンで参加して、みんなのbotに混じって遊べます。

仕組みは数当ての公開と同じ「Gradio ← `requests` → アリーナサーバ」。ボタンを押すと `/move` をサーバへ送り、`/state` を取り直して盤面を絵にして表示します。

- 盤面を絵に描く `render()` は用意してあります（`PIL`）
- あなたが書くのは、ボタンが押されたときに サーバへ動きを送る 1行です

先に「接続設定」（`SERVER` と `api`）と、対戦相手（先生のサーバ / ソロ練習）を用意しておいてください。

In [ ]:
!pip install --quiet gradio

In [ ]:
import gradio as gr
import requests
from PIL import Image, ImageDraw

CELL = 16   # 1マスの大きさ（ピクセル）

def render(state):
    """盤面の状態を絵にする"""
    W, H = state["w"], state["h"]
    img = Image.new("RGB", (W * CELL, H * CELL), "#161b22")
    d = ImageDraw.Draw(img)
    for i in range(W + 1):                             # 格子線
        d.line([(i*CELL, 0), (i*CELL, H*CELL)], fill="#2a2f3a")
    for j in range(H + 1):
        d.line([(0, j*CELL), (W*CELL, j*CELL)], fill="#2a2f3a")
    for x, y in state["coins"]:                        # コイン（黄）
        d.ellipse([x*CELL+4, y*CELL+4, x*CELL+CELL-4, y*CELL+CELL-4], fill="#f1c40f")
    for p in state["players"]:                         # プレイヤー（各色）
        x, y = p["x"], p["y"]
        d.ellipse([x*CELL+2, y*CELL+2, x*CELL+CELL-2, y*CELL+CELL-2], fill=p["color"])
    return img

my_id = None

def join_game(name):
    global my_id
    my_id = requests.post(SERVER + "/join", json={"name": name}).json()["id"]   # 参加して自分のidをもらう
    return render(requests.get(SERVER + "/state").json()), "参加しました（id={}）".format(my_id)

def move(direction):
    if my_id is None:
        return None, "先に「参加する」を押してください"

    # ここで、direction 方向へ /move する（requests で SERVER に送る）


    state = requests.get(SERVER + "/state").json()     # 最新の盤面を取り直す
    me = next((p for p in state["players"] if p["id"] == my_id), None)
    return render(state), "あなたのスコア: {}".format(me["score"] if me else 0)

# --- 操作パネルの見た目 ---
with gr.Blocks() as demo:
    gr.Markdown("## アリーナ操作パネル（人間プレイヤー）")
    name_in = gr.Textbox(label="名前", value="人間")
    join_btn = gr.Button("参加する", variant="primary")
    board = gr.Image(label="盤面")
    info = gr.Textbox(label="状態")
    up = gr.Button("↑ 上")
    with gr.Row():
        left = gr.Button("← 左"); down = gr.Button("↓ 下"); right = gr.Button("→ 右")

    join_btn.click(join_game, inputs=name_in, outputs=[board, info])
    up.click(lambda: move("up"),       outputs=[board, info])
    down.click(lambda: move("down"),   outputs=[board, info])
    left.click(lambda: move("left"),   outputs=[board, info])
    right.click(lambda: move("right"), outputs=[board, info])

demo.launch(share=True)

---
# 不正（チート）とその対策

今のアリーナには「本人確認」がありません。`id` を知っていれば誰でも他人を動かせます。ここからは対策を1つずつ積み上げていきます。攻撃を試す → 守りを足す → その守りの穴を突く → さらに守りを足す、の繰り返しです。

## 攻撃：なりすまし移動

`id` さえ分かれば、他人になりすまして動かせてしまいます。試しに `id=0` のプレイヤーを勝手に動かしてみます。

In [ ]:
# 他人(id=0)を勝手に動かす。本人確認が無いので通ってしまう
print(requests.post(SERVER + "/move", json={"id": 0, "dir": "up"}).json())

## 対策の1歩目：共通鍵で本人確認する

サーバが `join` のとき、プレイヤーごとに 共通鍵 を発行して渡します（本人とサーバだけが知る合言葉）。`move` には共通鍵を添えて送り、サーバは覚えている鍵と一致するかを確かめます。鍵を知らない人の `move` は拒否されます（403）。

まずこの仕組みだけを入れたサーバを起動します。

In [ ]:
# 共通鍵で本人確認するサーバ
import json, threading, time, random, requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class KeyServer(ThreadingHTTPServer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.keys = {}       # id -> 共通鍵
        self.next_id = 0

class KeyHandler(BaseHTTPRequestHandler):
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode(); self.send_response(code)
        self.send_header("Content-Type", "application/json"); self.end_headers(); self.wfile.write(b)
    def _body(self):
        n = int(self.headers.get("Content-Length", 0)); return json.loads(self.rfile.read(n) or b"{}")
    def do_POST(self):
        s = self.server; d = self._body()
        if self.path == "/join":
            i = s.next_id; s.next_id += 1
            key = "%08x" % random.randint(0, 0xffffffff)   # 共通鍵を発行
            s.keys[i] = key
            self._send({"id": i, "key": key})
        elif self.path == "/move":
            i = d.get("id")
            if s.keys.get(i) is None:
                self._send({"error": "unknown id"}, 400); return
            if d.get("key") != s.keys[i]:                  # 鍵が一致するか確認
                self._send({"error": "bad key"}, 403); return
            self._send({"ok": True})
        else:
            self._send({"error": "not found"}, 404)
    def log_message(self, *a): pass

_ks = KeyServer(("127.0.0.1", 0), KeyHandler)
threading.Thread(target=_ks.serve_forever, daemon=True).start()
time.sleep(0.3)
AUTH = "http://127.0.0.1:%d" % _ks.server_address[1]
print("共通鍵サーバ:", AUTH)

In [ ]:
# 例題：参加して共通鍵をもらい、鍵を添えて動く
me = requests.post(AUTH + "/join", json={}).json()
pid = me["id"]
key = me["key"]          # 自分とサーバだけが知る共通鍵
print("自分の鍵:", key)

print(requests.post(AUTH + "/move", json={"id": pid, "dir": "up", "key": key}).json())

In [ ]:
# 演習：わざと間違った鍵で move を送り、結果コードが 403 になることを確かめよう
#   requests.post(...).status_code で結果コードが取れます

## 攻撃：盗聴で共通鍵が盗まれる

本人確認はできましたが、大きな穴があります。`move` のたびに共通鍵をそのまま送っているので、通信を覗いた攻撃者には鍵が丸見えです。盗んだ鍵を使えば、なりすましは簡単です。

In [ ]:
# 通信には毎回こういうJSONが流れている。覗けば key が丸見え
print({"id": pid, "dir": "up", "key": key})

# 攻撃者：盗んだ鍵でなりすまし。通ってしまう
stolen = key
print(requests.post(AUTH + "/move", json={"id": pid, "dir": "down", "key": stolen}).json())

## 対策の2歩目：鍵は送らず、ハッシュで署名する

共通鍵そのものは二度と送りません。かわりに「共通鍵＋動きの内容」をつないだ文字列の ハッシュ を計算し、署名 として送ります。サーバは自分が覚えている鍵で同じ計算をして照合します。

- 攻撃者は鍵を知らないので、正しい署名を作れない（なりすまし防止）
- 動きの内容が1文字でも変われば署名も変わる（改ざん検知）

この仕組みに変えたサーバを起動します。

In [ ]:
# 署名つきの実験サーバ
import json, threading, time, random, hashlib, requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class SecureServer(ThreadingHTTPServer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.keys = {}       # id -> 共通鍵
        self.next_id = 0

class SecureHandler(BaseHTTPRequestHandler):
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode(); self.send_response(code)
        self.send_header("Content-Type", "application/json"); self.end_headers(); self.wfile.write(b)
    def _body(self):
        n = int(self.headers.get("Content-Length", 0)); return json.loads(self.rfile.read(n) or b"{}")
    def do_POST(self):
        s = self.server; d = self._body()
        if self.path == "/join":
            i = s.next_id; s.next_id += 1
            key = "%08x" % random.randint(0, 0xffffffff)
            s.keys[i] = key
            self._send({"id": i, "key": key})
        elif self.path == "/move":
            i = d.get("id"); key = s.keys.get(i)
            if key is None:
                self._send({"error": "unknown id"}, 400); return
            # 署名の照合（共通鍵は送られてこない）
            want = hashlib.sha256("{}|{}|{}".format(key, i, d.get("dir")).encode()).hexdigest()
            if d.get("sig") != want:
                self._send({"error": "bad signature"}, 403); return
            self._send({"ok": True})
        else:
            self._send({"error": "not found"}, 404)
    def log_message(self, *a): pass

_ss = SecureServer(("127.0.0.1", 0), SecureHandler)
threading.Thread(target=_ss.serve_forever, daemon=True).start()
time.sleep(0.3)
SECURE = "http://127.0.0.1:%d" % _ss.server_address[1]
print("署名つきサーバ:", SECURE)

## 署名つきで動く

参加して共通鍵をもらい、move ごとに `sha256(共通鍵|id|向き)` の署名を付けて送ります。今度は通信に流れるのは署名だけで、鍵は流れません。署名を作る部分を書きましょう。

In [ ]:
import hashlib

me = requests.post(SECURE + "/join", json={}).json()
pid = me["id"]
key = me["key"]

direction = "up"

# 演習：sha256(f"{key}|{pid}|{direction}") の16進文字列を sig に入れよう
#   hashlib が使えます（16進文字列は .hexdigest() で得られます）


print(requests.post(SECURE + "/move", json={"id": pid, "dir": direction, "sig": sig}).json())

## なりすましは弾かれる

通信を覗いても流れているのは署名だけです。署名から鍵は逆算できず（ハッシュの不可逆性）、鍵が無いと新しい署名は作れません。でたらめな署名で送ると拒否されます（403）。

In [ ]:
# 鍵を知らずに、でたらめな署名でなりすまし
fake = "0" * 64
print("結果コード:", requests.post(SECURE + "/move", json={"id": pid, "dir": "up", "sig": fake}).status_code, "（403なら拒否成功）")

## 盗聴対策：通信を暗号化する（XOR暗号）

署名はなりすましと改ざんを防ぎますが、通信の中身（どの向きに動いたか）は覗けばそのまま読めます。中身も隠したいときは、セキュリティ回で学んだ 共通鍵での暗号化 を使います。ここでは一番シンプルな XOR暗号 を使います。

同じ鍵でもう一度XORすると元に戻る、という性質がポイントです。

In [ ]:
def xor_cipher(data, key):
    return bytes(b ^ key for b in data)

key = 0x5A                       # 共通鍵（1バイト）
plain = "up".encode()            # 送りたい内容

enc = xor_cipher(plain, key)     # 暗号化して送る
print("暗号文:", enc.hex())      # 通信を覗いてもこれしか見えない
print("復号 :", xor_cipher(enc, key).decode())   # 受け取った側は同じ鍵で戻す

In [ ]:
# 演習：受け取った暗号文 secret を、共通鍵 key で復号して向きを取り出そう
#   上の xor_cipher がそのまま使えます
key = 0x5A
secret = bytes.fromhex("3e352d34")   # ある向きを暗号化したもの

# ここに復号して direction（文字列）を求めるコードを書く


print("隠されていた向き:", direction)

## 発展：公開鍵で署名する（RSA）

共通鍵方式は「サーバと本人が同じ鍵を共有」します。公開鍵署名なら、bot が 秘密鍵 で署名し、サーバは 公開鍵 で検証するので、鍵を共有せずに本人確認できます。セキュリティ回の手作りRSAをそのまま使います。

In [ ]:
import hashlib

# 鍵（セキュリティ回と同じ手作りRSA）
p, q = 61, 53
N = p * q
e = 17
d = pow(e, -1, (p - 1) * (q - 1))     # 秘密鍵

move = "up"
digest = int(hashlib.sha256(move.encode()).hexdigest(), 16) % N
signature = pow(digest, d, N)          # 秘密鍵で署名

# サーバ役：公開鍵 (e, N) で検証
print("本人の署名は正しい:", pow(signature, e, N) == digest)
tampered = int(hashlib.sha256("down".encode()).hexdigest(), 16) % N
print("改ざんは検出される:", pow(signature, e, N) != tampered)

In [ ]:
# 演習：別の向き "left" に秘密鍵 d で署名して sig2 を作り、公開鍵 (e, N) で検証しよう
#   上の p, q, N, e, d と digest の作り方をそのまま使う
move2 = "left"

# digest2 = ...（move2 のハッシュを N で割った余り）
# sig2 = ...（秘密鍵 d で署名）


print("署名が正しい:", pow(sig2, e, N) == digest2)

## まとめ

- なりすまし → 共通鍵で本人確認する
- 盗聴で鍵が盗まれる → 鍵は送らず、ハッシュ署名にする
- 改ざん → 内容が変われば署名も変わるので検出できる
- 通信の中身も隠したい → 共通鍵での暗号化（XOR暗号）
- 鍵の共有をなくしたい → 公開鍵署名（RSA）

「攻撃を知り、穴をひとつずつ塞ぐ」。これがセキュリティ設計の基本です。


## 総合演習：アリーナサーバをセキュリティ強化する

最初に使ったアリーナサーバは `/move` に `id` と `dir` を送るだけで、本人確認がありませんでした。ここまでの 共通鍵とハッシュ署名 を組み込んだ強化版サーバを用意しました。`/join` で共通鍵を配り、`/move` で署名を照合し、通れば盤上のコマを動かします（起動するだけ）。

あなたの仕事は クライアント側 です。次の3つを確かめるコードを書いてください。

1. 正しく署名した move が通り、新しい位置（`pos`）が返る
2. 共通鍵を知らないなりすまし（でたらめな署名）が弾かれる（403）
3. 署名はそのままで向きだけ書き換えた改ざんが弾かれる（403）

In [ ]:
# 強化版アリーナサーバ（署名で本人確認）
import json, threading, time, random, hashlib, requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class SecureArena(ThreadingHTTPServer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.keys = {}       # id -> 共通鍵
        self.pos = {}        # id -> [x, y]
        self.next_id = 0

def _sign(key, i, direction):
    return hashlib.sha256("{}|{}|{}".format(key, i, direction).encode()).hexdigest()

class SArenaHandler(BaseHTTPRequestHandler):
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode(); self.send_response(code)
        self.send_header("Content-Type", "application/json"); self.end_headers(); self.wfile.write(b)
    def _body(self):
        n = int(self.headers.get("Content-Length", 0)); return json.loads(self.rfile.read(n) or b"{}")
    def do_POST(self):
        s = self.server; d = self._body()
        if self.path == "/join":
            i = s.next_id; s.next_id += 1
            key = "%08x" % random.randint(0, 0xffffffff)
            s.keys[i] = key; s.pos[i] = [12, 12]
            self._send({"id": i, "key": key, "pos": s.pos[i]})
        elif self.path == "/move":
            i = d.get("id"); key = s.keys.get(i)
            if key is None:
                self._send({"error": "unknown id"}, 400); return
            if d.get("sig") != _sign(key, i, d.get("dir")):
                self._send({"error": "bad signature"}, 403); return
            x, y = s.pos[i]
            dx = {"left": -1, "right": 1}.get(d.get("dir"), 0)
            dy = {"up": -1, "down": 1}.get(d.get("dir"), 0)
            s.pos[i] = [max(0, min(23, x + dx)), max(0, min(23, y + dy))]
            self._send({"ok": True, "pos": s.pos[i]})
        else:
            self._send({"error": "not found"}, 404)
    def log_message(self, *a): pass

_sa = SecureArena(("127.0.0.1", 0), SArenaHandler)
threading.Thread(target=_sa.serve_forever, daemon=True).start()
time.sleep(0.3)
ARENA2 = "http://127.0.0.1:%d" % _sa.server_address[1]
print("強化版アリーナ:", ARENA2)

In [ ]:
import hashlib

# 総合演習：強化版アリーナ(ARENA2)に参加し、共通鍵で署名して安全に動こう
# 1) 参加して id と 共通鍵 key を受け取る
# 2) 署名を作る関数 sign(key, i, direction) を用意する（sha256 の16進文字列）
# 3) 次の3つを確かめる
#    (a) 正しい署名の move が通る（ok と新しい pos が返る）
#    (b) 共通鍵を知らないなりすまし（でたらめな署名）が 403 になる
#    (c) 署名はそのまま dir だけ書き換えた改ざんが 403 になる

---
# おわりに

ここまでで、あなたは自分の手で サーバを立て、通信し、botを動かし、全員で競い、そして不正と対策まで 体験しました。

- クラスでデータの設計図を作り
- バイナリとセキュリティでデータの正体と守り方を知り
- ネットワークで通信の仕組みを作り
- そして今日、それらを 全部つないで 動くものを作りました

ここからは、ルールを変える・戦略を磨く・不正対策を実装する——好きに拡張してみてください。おつかれさまでした。